In [ ]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from uncertainties import ufloat as uf
from scipy.optimize import curve_fit
from uncertainties import correlated_values
from pathlib import Path

# path config
downloads_dir = Path.home() / 'Downloads'
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkcí

# nejistota typu A 
def unc_A(data):
    """
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    """
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

# nacteni z df do jednotlivych arrays
def df_to_arrays(df, sloupce, od=0, do=None, dtype=float):
    if isinstance(sloupce, str):
        sloupce = [sloupce]

    chybejici = [s for s in sloupce if s not in df.columns]
    if chybejici:
        raise KeyError(f"Chybejici sloupce: {chybejici}")

    return [df[s].iloc[od:do].dropna().to_numpy(dtype=dtype) for s in sloupce]

# ufloat z aritmetickeho prumeru
def ufloat_unc_A(array):
    return uf(np.mean(array), unc_A(array))

# prepocet z popt a pcov na ufloaty
def popt_pcov_to_ufloat(popt, pcov, correlated=False):
    """
    Převede výsledek curve_fit (popt, pcov) na ufloaty.

    Parametry
    ---------
    popt : array_like   optimální parametry z curve_fit
    pcov : array_like   kovarianční matice z curve_fit
    correlated : bool   True  -> ufloaty nesou korelace mezi parametry
                        False -> nezávislé ufloaty (jen sqrt diagonály pcov)

    Vrací
    -----
    tuple[ufloat, ...]  parametry s nejistotami
    """
    popt = np.asarray(popt, dtype=float)
    pcov = np.asarray(pcov, dtype=float)

    if correlated:
        return tuple(correlated_values(popt, pcov))

    perr = np.sqrt(np.diag(pcov))
    return tuple(uf(v, e) for v, e in zip(popt, perr))

# array do latex
def array_to_latex(*array, dig=1, format='u', style='L', only_return=False):
    output = [f'${u:.{dig}{format}{style}}$' for u in array]
    if only_return:
        return output
    else:
        for s in output:
            print(s)

In [ ]:
# nacteni google tabulek
sheet_id = "1cXKdaqbpzERg_syz5NUfd-MyWxmNyKs_Tjba6bure2k"
gid = "2139038384"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_peaks = pd.read_csv(url)

In [ ]:
# peaky pro theta
N_zavit = 1000
L_kyvadlo = 1.7
R_civka = 40.0
mu_0 = 1.25663706127*1e-6
g_tiha = 9.81
theta_max = np.arange(2,13)*np.pi/180

# bez 11_min a 5_min
t_5_min = [1.004300]
U_5_min = [-0.621071]
t_11_min = [0.929500]
U_11_min = [-1.26663]
t_12_max = [0.800800]
U_12_max = [1.296594]
t_12_min = [0.821800]
U_12_min = [-1.253414]

df_peaks_theta_list = ['fm10.max_t', 'fm10.max_U', 'fm10.min_t', 'fm10.min_U', 'fm11.max_t', 'fm11.max_U', 'fm2.max_t', 'fm2.max_U', 'fm2.min_t', 'fm2.min_U', 'fm3.max_t', 'fm3.max_U', 'fm3.min_t', 'fm3.min_U', 'fm4.max_t', 'fm4.max_U', 'fm4.min_t', 'fm4.min_U', 'fm5.max_t', 'fm5.max_U', 'fm6.max_t', 'fm6.max_U', 'fm6.min_t', 'fm6.min_U', 'fm7.max_t', 'fm7.max_U', 'fm7.min_t', 'fm7.min_U', 'fm8.max_t', 'fm8.max_U', 'fm8.min_t', 'fm8.min_U', 'fm9.max_t', 'fm9.max_U', 'fm9.min_t', 'fm9.min_U']
t_10_max, U_10_max, t_10_min, U_10_min, t_11_max, U_11_max, t_2_max, U_2_max, t_2_min, U_2_min, t_3_max, U_3_max, t_3_min, U_3_min, t_4_max, U_4_max, t_4_min, U_4_min, t_5_max, U_5_max, t_6_max, U_6_max, t_6_min, U_6_min, t_7_max, U_7_max, t_7_min, U_7_min, t_8_max, U_8_max, t_8_min, U_8_min, t_9_max, U_9_max, t_9_min, U_9_min  = df_to_arrays(df_peaks, df_peaks_theta_list)
t_maxima = [t_2_max, t_3_max, t_4_max, t_5_max, t_6_max, t_7_max, t_8_max, t_9_max, t_10_max, t_11_max, t_12_max]
t_max = np.array([x[0] for x in t_maxima])
t_minima = [t_2_min, t_3_min, t_4_min, t_5_min, t_6_min, t_7_min, t_8_min, t_9_min, t_10_min, t_11_min, t_12_min]
t_min = np.array([x[0] for x in t_minima])

U_maxima = [U_2_max, U_3_max, U_4_max, U_5_max, U_6_max, U_7_max, U_8_max, U_9_max, U_10_max, U_11_max, U_12_max]
U_max = np.array([x[0] for x in U_maxima])
U_minima = [U_2_min, U_3_min, U_4_min, U_5_min, U_6_min, U_7_min, U_8_min, U_9_min, U_10_min, U_11_min, U_12_min]
U_min = np.array([x[0] for x in U_minima])

delta_t = abs(t_min - t_max)
delta_U = abs(U_max - U_min)

v_max = ((g_tiha*L_kyvadlo)**0.5)*theta_max
v_max_sin = 2*((g_tiha*L_kyvadlo)**0.5)*np.sin(theta_max/2)
a_civka = delta_t*v_max
a_civka_sin = delta_t*v_max_sin
print(f'Efektivní poloměr cívky je:\n{ufloat_unc_A(a_civka)*100:.1uP} cm\nBez aproximace sinu:\n{ufloat_unc_A(a_civka_sin)*100:.2uP} cm')
mag_moment = (U_max*(25*(5**0.5)/24)*((a_civka**2)/(N_zavit*mu_0)))/v_max
print(f'Odhad magnetického dipólového momentu je:\n'f'{ufloat_unc_A(mag_moment):.1uP} Wb.m')
U_max_model = (24/(25*(5*0.5)))*((N_zavit*mu_0*mag_moment)/(a_civka**2))*v_max

def lin(x,A):
    return A*x

def inverse(x, A):
    return A/x

fig, ax = plt.subplots(1, 2, figsize=(16, 9))
ax[0].set_ylabel(r'$U_{\mathrm{max}}$ [V]', fontsize=20)
ax[0].set_xlabel(r'$\vartheta$ [°]', fontsize=20)
ax[1].set_ylabel(r'$\Delta t$ [s]', fontsize=20)
ax[1].set_xlabel(r'$\vartheta$ [°]', fontsize=20)

ax[0].scatter(theta_max*180/np.pi, U_max)
popt, pcov = curve_fit(lin, theta_max, U_max)
ax[0].plot(theta_max*180/np.pi, lin(theta_max, *popt), label=r'Fit funkcí: $A*x$''\n'f'A = {popt[0]:.5f}\n')

ax[1].scatter(theta_max*180/np.pi, delta_t)
popt, pcov = curve_fit(inverse, theta_max, delta_t)
theta_max_linspace = np.linspace(np.min(theta_max), np.max(theta_max), 1000)
ax[1].plot(theta_max_linspace*180/np.pi, inverse(theta_max_linspace, *popt), label=r'Fit funkcí: $A/x$''\n'f'A = {popt[0]:.5f}')

ax[0].tick_params(labelsize=15)
ax[0].grid(True, alpha=0.7)
ax[0].legend(fontsize=15)
ax[1].tick_params(labelsize=15)
ax[1].grid(True, alpha=0.7)
ax[1].legend(fontsize=15)
plt.savefig(downloads_dir / 'aproximace_U_theta.pdf', dpi=300, bbox_inches='tight')
for u in delta_U:
    print(f'{u:.3f}')

In [ ]:
peaky_t = {
    '20': [0.8198, 2.0949, 3.4402, 4.688, 6.0425, 6.0458, 6.0542, 7.2389, 8.6159, 9.5603],
    '40': [0.7722, 2.0537, 3.3913, 4.6636, 6.0108, 6.0112, 7.2582, 7.2584, 8.6109, 9.8336, 9.8385, 11.2248, 12.3306, 12.3482, 13.7467, 13.747],
    '60': [0.6307, 1.911, 3.2487, 3.249, 4.5228, 5.8661, 7.1291, 8.4811, 9.7194, 11.0913, 11.0918, 12.2896, 13.6787, 14.7914, 16.2526],
    '80': [0.7791, 2.0616, 3.398, 4.6742, 6.0156, 7.2837, 8.6302, 9.8877, 11.2425, 12.4817, 13.8531, 13.8535, 15.0316, 15.0349, 16.431499, 17.554001, 17.554199, 19.001499, 19.014, 19.9457, 20.006399, 20.006701, 20.007099],
    '100': [0.7471, 2.0297, 3.3665, 4.6437, 5.9849, 7.2544, 8.6007, 9.86, 11.2151, 12.4595, 13.8224, 15.0457, 16.425699, 17.599501, 19.0347, 20.113001, 21.6982, 22.538],
    '120': [0.6397, 1.9225, 3.2597, 4.5367, 5.8792, 7.1483, 8.496, 9.7594, 9.7597, 11.1107, 12.3623, 13.7189, 13.7191, 14.9589, 16.3339, 17.534401, 18.936501, 20.0926, 21.5369, 22.6173, 24.1737, 25.1166],
    '150': [0.4355, 1.7185, 3.0547, 4.3341, 5.6727, 6.9479, 8.2919, 9.5596, 10.9078, 12.1652, 13.5216, 14.7696, 16.129999, 17.3629, 17.3631, 18.740999, 19.948299, 21.3454, 22.502199, 23.9398, 25.060801, 26.524],
    '200': [0.63, 1.9144, 3.2498, 4.5306, 5.8684, 7.1463, 8.4871, 9.7593, 11.1044, 12.371, 13.7193, 14.9793, 16.3342, 17.5795, 17.5798, 18.943199, 20.179701, 21.552601, 21.552799, 22.778099, 24.161501, 25.3552, 26.762899, 27.9209, 29.3661],
    '1k': [0.6573, 1.9431, 3.2778, 4.5606, 5.8965, 7.1781, 8.5151, 9.7955, 11.1343, 12.413, 13.7537, 15.0303, 16.3731, 17.6465, 18.991199, 20.2624, 21.6085, 22.876801, 24.224501, 25.4893, 26.841101, 28.104, 29.4573, 30.7159, 32.070599, 33.3279, 34.684101, 35.935501, 37.297798, 38.5448, 38.544998, 39.911098, 41.154701, 41.157001, 41.157501, 42.52, 43.760399, 45.129101, 45.1371, 46.366699, 47.743301, 48.973, 50.354, 51.575401, 52.9631, 54.168499, 55.562401, 56.769699, 58.169102, 59.369099, 60.7631, 60.770401, 61.948299, 63.364601, 64.565201, 65.975304, 67.1576, 68.584801, 69.756104, 71.172401, 72.350601, 73.815697, 74.963097, 76.449799, 77.553398, 79.112, 80.171303, 81.696999, 82.741402, 84.332497, 85.3396, 86.940102, 87.8825, 87.915703, 89.514702],
    '1m': [0.868, 2.1535, 3.4882, 4.7716, 6.108, 7.39, 8.7268, 10.0084, 11.3458, 12.6267, 13.9651, 15.2446, 16.584499, 17.863001, 19.203899, 20.4813, 21.822901, 23.098101, 24.4412, 25.7157, 27.0599, 28.3321, 29.6777, 30.947701, 32.295101, 33.562599, 33.563, 34.912701, 36.180199, 37.528099, 37.5284, 37.528702, 38.7938, 40.1441, 41.4081, 42.759701, 44.021301, 45.374802, 46.634602, 47.988998, 49.247799, 50.605, 51.861099, 53.219501, 54.4729, 55.831299, 57.0849, 58.446602, 59.696602, 61.060699, 62.307098, 63.672798, 64.915802, 66.281799, 67.530098, 68.896202, 70.134399, 71.509003, 72.745903, 74.118698, 75.357803, 76.725601, 77.959801, 79.3424, 80.571404, 81.949203, 83.176498, 84.557701, 84.557899, 85.784599, 87.166199, 88.392403, 89.779701, 90.999496, 92.384499, 93.601997, 94.989799, 96.203796, 97.597198, 98.8106, 100.203697, 100.204102, 101.412399, 102.807404, 104.0121, 105.408897, 105.415497, 105.415802, 106.615303, 108.022301, 109.210999, 109.216904, 110.613602, 111.822998, 111.823196, 113.220703, 114.414803, 115.823502, 117.017303, 117.0177, 118.423798, 118.430603, 119.6157, 121.024803, 122.223297, 123.634499, 124.812202, 124.812401, 124.8209, 124.824402, 126.239098, 127.410797, 128.842804, 130.026794, 131.446899, 132.612595, 134.039093, 134.052399, 135.229004, 136.639801, 137.819107, 139.266907, 139.267303, 140.444901, 140.448105, 141.873703, 143.039398, 143.039795, 143.0401, 143.047897, 144.498093, 145.669998, 147.123398, 148.264603, 149.724304],
}
pt20 = np.array(peaky_t['20'])
pt40 = np.array(peaky_t['40'])
pt60 = np.array(peaky_t['60'])
pt80 = np.array(peaky_t['80'])
pt100 = np.array(peaky_t['100'])
pt120 = np.array(peaky_t['120'])
pt150 = np.array(peaky_t['150'])
pt200 = np.array(peaky_t['200'])
pt1k = np.array(peaky_t['1k'])
pt1m = np.array(peaky_t['1m'])

peaky_U = {
    '20': [0.319374, 0.270382, 0.220382, 0.170383, 0.130387, 0.130387, 0.130387, 0.084208, 0.047536, 0.020391],
    '40': [0.539775, 0.488944, 0.421175, 0.360157, 0.310149, 0.310149, 0.250151, 0.250151, 0.200156, 0.160149, 0.160149, 0.110161, 0.060159, 0.060159, 0.030164, 0.030164],
    '60': [0.639793, 0.56025, 0.510232, 0.510232, 0.458476, 0.400245, 0.36024, 0.303576, 0.250244, 0.210247, 0.210247, 0.16025, 0.11025, 0.070244, 0.040251],
    '80': [0.709973, 0.650071, 0.593915, 0.540101, 0.490094, 0.440095, 0.390108, 0.350058, 0.300102, 0.250104, 0.210104, 0.210104, 0.160102, 0.160102, 0.120107, 0.080102, 0.080102, 0.050107, 0.050107, 0.020105, 0.020105, 0.020105, 0.020105],
    '100': [0.755516, 0.709882, 0.658573, 0.589913, 0.546652, 0.494032, 0.449772, 0.403599, 0.359924, 0.319913, 0.269918, 0.228988, 0.189918, 0.148881, 0.109924, 0.069919, 0.049924, 0.019924],
    '120': [0.807124, 0.751902, 0.700776, 0.632457, 0.590785, 0.540783, 0.500751, 0.460779, 0.460779, 0.410783, 0.378343, 0.330785, 0.330785, 0.290777, 0.250784, 0.21078, 0.18025, 0.130786, 0.110248, 0.070104, 0.050788, 0.020787],
    '150': [0.840243, 0.790599, 0.750415, 0.700078, 0.650608, 0.610611, 0.568968, 0.525589, 0.484472, 0.440599, 0.406458, 0.365846, 0.330609, 0.290609, 0.290609, 0.260612, 0.220604, 0.190609, 0.150608, 0.127185, 0.08545, 0.07061],
    '200': [0.910593, 0.869067, 0.820944, 0.770699, 0.730642, 0.680961, 0.646734, 0.610923, 0.570947, 0.540875, 0.500965, 0.469051, 0.430973, 0.390977, 0.390977, 0.360968, 0.330959, 0.30097, 0.30097, 0.267633, 0.240966, 0.20097, 0.180969, 0.150961, 0.130966],
    '1k': [1.062028, 1.051601, 1.012201, 0.962458, 0.942542, 0.911424, 0.88267, 0.870346, 0.84222, 0.822669, 0.8015, 0.781937, 0.752674, 0.732668, 0.712667, 0.692211, 0.672645, 0.652673, 0.632676, 0.612673, 0.602298, 0.582668, 0.562671, 0.54554, 0.532665, 0.512676, 0.502663, 0.482677, 0.472672, 0.452672, 0.452672, 0.442664, 0.422672, 0.422672, 0.422672, 0.412673, 0.392675, 0.382677, 0.382677, 0.370645, 0.362674, 0.342682, 0.332678, 0.314977, 0.312681, 0.292674, 0.287628, 0.272628, 0.262675, 0.243618, 0.242673, 0.242673, 0.222675, 0.222682, 0.202675, 0.20268, 0.182681, 0.182676, 0.162679, 0.16268, 0.142674, 0.148219, 0.122681, 0.132676, 0.102677, 0.112678, 0.092499, 0.102677, 0.072684, 0.090776, 0.062591, 0.08206, 0.04268, 0.04268, 0.072473],
    '1m': [1.102477, 1.1028, 1.07295, 1.036375, 1.021315, 0.991939, 0.972309, 0.952641, 0.932452, 0.915313, 0.893104, 0.883109, 0.863095, 0.852422, 0.833057, 0.813108, 0.802661, 0.783107, 0.770353, 0.753108, 0.742946, 0.73255, 0.713111, 0.703103, 0.693081, 0.673116, 0.673116, 0.664137, 0.653112, 0.643107, 0.643107, 0.643107, 0.633106, 0.623083, 0.612446, 0.603098, 0.592955, 0.582742, 0.570566, 0.563097, 0.55311, 0.543109, 0.533099, 0.523109, 0.513105, 0.5078, 0.499985, 0.493102, 0.483111, 0.473112, 0.465517, 0.463111, 0.45311, 0.443113, 0.433117, 0.433112, 0.423112, 0.422835, 0.410941, 0.403676, 0.393111, 0.393111, 0.383112, 0.383108, 0.373115, 0.37303, 0.363115, 0.363113, 0.363113, 0.352806, 0.349557, 0.341836, 0.341931, 0.332293, 0.330181, 0.322741, 0.322579, 0.311243, 0.312872, 0.30306, 0.303112, 0.303112, 0.293086, 0.29311, 0.28311, 0.283114, 0.283114, 0.283114, 0.273111, 0.273121, 0.263109, 0.263109, 0.263114, 0.253113, 0.253113, 0.258491, 0.243672, 0.253121, 0.243109, 0.243109, 0.243113, 0.243113, 0.233109, 0.233114, 0.223111, 0.231885, 0.213116, 0.213116, 0.213116, 0.213116, 0.223119, 0.208816, 0.213114, 0.203111, 0.211874, 0.193113, 0.203114, 0.203114, 0.190584, 0.193117, 0.183121, 0.193112, 0.193112, 0.173113, 0.173113, 0.183113, 0.17311, 0.17311, 0.17311, 0.17311, 0.183111, 0.163114, 0.173115, 0.158163, 0.172861],
}
p20 = (np.array(peaky_U['20'])*(20/(20+R_civka))**(-1))
p40 = (np.array(peaky_U['40'])*(40/(40+R_civka))**(-1))
p60 = (np.array(peaky_U['60'])*(60/(60+R_civka))**(-1))
p80 = (np.array(peaky_U['80'])*(80/(80+R_civka))**(-1))
p100 = (np.array(peaky_U['100'])*(100/(100+R_civka))**(-1))
p120 = (np.array(peaky_U['120'])*(120/(120+R_civka))**(-1))
p150 = (np.array(peaky_U['150'])*(150/(150+R_civka))**(-1))
p200 = (np.array(peaky_U['200'])*(200/(200+R_civka))**(-1))
p1k = (np.array(peaky_U['1k'])*(1e3/(1e3+R_civka))**(-1))
p1m = (np.array(peaky_U['1m'])*(1e6/(1e6+R_civka))**(-1))

nenormovaci_konstanta = 1
p20 = p20*(p200[0]/p20[0])*nenormovaci_konstanta
p40 = p40*(p200[0]/p40[0])*nenormovaci_konstanta
p60 = p60*(p200[0]/p60[0])*nenormovaci_konstanta
p80 = p80*(p200[0]/p80[0])*nenormovaci_konstanta
p100 = p100*(p200[0]/p100[0])*nenormovaci_konstanta
p120 = p120*(p200[0]/p120[0])*nenormovaci_konstanta
p150 = p150*(p200[0]/p150[0])*nenormovaci_konstanta
p200 = p200*nenormovaci_konstanta

In [ ]:
# tlumeni

df_peaks_peak_list = ['fmo100.max_t', 'fmo100.max_U', 'fmo100.min_t', 'fmo100.min_U', 'fmo120.max_t', 'fmo120.max_U', 'fmo120.min_t', 'fmo120.min_U', 'fmo150.max_t', 'fmo150.max_U', 'fmo150.min_t', 'fmo150.min_U', 'fmo1k.max_t', 'fmo1k.max_U', 'fmo1k.min_t', 'fmo1k.min_U', 'fmo1m.max_t', 'fmo1m.max_U', 'fmo1m.min_t', 'fmo1m.min_U', 'fmo20.max_t', 'fmo20.max_U', 'fmo20.min_t', 'fmo20.min_U', 'fmo200.max_t', 'fmo200.max_U', 'fmo200.min_t', 'fmo200.min_U', 'fmo40.max_t', 'fmo40.max_U', 'fmo40.min_t', 'fmo40.min_U', 'fmo60.max_t', 'fmo60.max_U', 'fmo60.min_t', 'fmo60.min_U', 'fmo80.max_t', 'fmo80.max_U', 'fmo80.min_t', 'fmo80.min_U', 'fmoo20.max_t', 'fmoo20.max_U', 'fmoo20.min_t', 'fmoo20.min_U']
odpor_100_max_t, odpor_100_max_U, odpor_100_min_t, odpor_100_min_U, odpor_120_max_t, odpor_120_max_U, odpor_120_min_t, odpor_120_min_U, odpor_150_max_t, odpor_150_max_U, odpor_150_min_t, odpor_150_min_U, odpor_1k_max_t, odpor_1k_max_U, odpor_1k_min_t, odpor_1k_min_U, odpor_1m_max_t, odpor_1m_max_U, odpor_1m_min_t, odpor_1m_min_U, odpor_20_max_t, odpor_20_max_U, odpor_20_min_t, odpor_20_min_U, odpor_200_max_t, odpor_200_max_U, odpor_200_min_t, odpor_200_min_U, odpor_40_max_t, odpor_40_max_U, odpor_40_min_t, odpor_40_min_U, odpor_60_max_t, odpor_60_max_U, odpor_60_min_t, odpor_60_min_U, odpor_80_max_t, odpor_80_max_U, odpor_80_min_t, odpor_80_min_U, oodpor_20_max_t, oodpor_20_max_U, oodpor_20_min_t, oodpor_20_min_U = df_to_arrays(df_peaks, df_peaks_peak_list)
oodpor_20_max_U, oodpor_20_min_U, odpor_40_max_U, odpor_40_min_U, odpor_60_max_U, odpor_60_min_U, odpor_80_max_U, odpor_80_min_U, odpor_100_max_U, odpor_100_min_U, odpor_120_max_U, odpor_120_min_U, odpor_150_max_U, odpor_150_min_U, odpor_200_max_U, odpor_200_min_U, odpor_1k_max_U, odpor_1k_min_U, odpor_1m_max_U, odpor_1m_min_U = oodpor_20_max_U*(20/(20+R_civka)), oodpor_20_min_U*(20/(20+R_civka)), odpor_40_max_U*(40/(40+R_civka)), odpor_40_min_U*(40/(40+R_civka)), odpor_60_max_U*(60/(60+R_civka)), odpor_60_min_U*(60/(60+R_civka)), odpor_80_max_U*(80/(80+R_civka)), odpor_80_min_U*(80/(80+R_civka)), odpor_100_max_U*(100/(100+R_civka)), odpor_100_min_U*(100/(100+R_civka)), odpor_120_max_U*(120/(120+R_civka)), odpor_120_min_U*(120/(120+R_civka)), odpor_150_max_U*(150/(150+R_civka)), odpor_150_min_U*(150/(150+R_civka)), odpor_200_max_U*(200/(200+R_civka)), odpor_200_min_U*(200/(200+R_civka)), odpor_1k_max_U*(1000/(1000+R_civka)), odpor_1k_min_U*(1000/(1000+R_civka)), odpor_1m_max_U*(1e6/(1e6+R_civka)), odpor_1m_min_U*(1e6/(1e6+R_civka))

def lin(x,A, B):
    return -A*x + B


#theta_max_20_odpor = (oodpor_20_max_U/((24/(25*(5*0.5)))*((N_zavit*mu_0*ufloat_unc_A(mag_moment))/(ufloat_unc_A(a_civka)**2))))/((g_tiha*L_kyvadlo)**0.5)
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'Maximum', fontsize=20)
ax.set_xlabel(r'$t$ [s]', fontsize=20)

alpha_list = []

ax.scatter(pt20, p20, label=r'$R = 20\,\Omega$')
popt, pcov= curve_fit(lin, pt20, p20)
ax.plot(pt20, lin(pt20, *popt))
print(f'Směrnice je:{popt_pcov_to_ufloat(popt, pcov)[0]:.1uP}')
alpha_list.append(popt_pcov_to_ufloat(popt, pcov)[0].n)

ax.scatter(pt40, p40, label=r'$R = 40\,\Omega$')
popt,pcov= curve_fit(lin, pt40, p40)
ax.plot(pt40, lin(pt40, *popt))
print(f'Směrnice je:{popt_pcov_to_ufloat(popt, pcov)[0]:.1uP}')
alpha_list.append(popt_pcov_to_ufloat(popt, pcov)[0].n)

ax.scatter(pt60, p60, label=r'$R = 60\,\Omega$')
popt,pcov= curve_fit(lin, pt60, p60)
ax.plot(pt60, lin(pt60, *popt))
print(f'Směrnice je:{popt_pcov_to_ufloat(popt, pcov)[0]:.1uP}')
alpha_list.append(popt_pcov_to_ufloat(popt, pcov)[0].n)

ax.scatter(pt80, p80, label=r'$R = 80\,\Omega$')
popt,pcov= curve_fit(lin, pt80, p80)
ax.plot(pt80, lin(pt80, *popt))
print(f'Směrnice je:{popt_pcov_to_ufloat(popt, pcov)[0]:.1uP}')
alpha_list.append(popt_pcov_to_ufloat(popt, pcov)[0].n)

ax.scatter(pt100, p100, label=r'$R = 100\,\Omega$')
popt,pcov= curve_fit(lin, pt100, p100)
ax.plot(pt100, lin(pt100, *popt))
print(f'Směrnice je:{popt_pcov_to_ufloat(popt, pcov)[0]:.1uP}')
alpha_list.append(popt_pcov_to_ufloat(popt, pcov)[0].n)

ax.scatter(pt120, p120, label=r'$R = 120\,\Omega$')
popt,pcov= curve_fit(lin, pt120, p120)
ax.plot(pt120, lin(pt120, *popt))
print(f'Směrnice je:{popt_pcov_to_ufloat(popt, pcov)[0]:.1uP}')
alpha_list.append(popt_pcov_to_ufloat(popt, pcov)[0].n)

ax.scatter(pt150, p150, label=r'$R = 150\,\Omega$')
popt,pcov= curve_fit(lin, pt150, p150)
ax.plot(pt150, lin(pt150, *popt))
print(f'Směrnice je:{popt_pcov_to_ufloat(popt, pcov)[0]:.1uP}')
alpha_list.append(popt_pcov_to_ufloat(popt, pcov)[0].n)

ax.scatter(pt200, p200, label=r'$R = 200\,\Omega$')
popt,pcov= curve_fit(lin, pt200, p200)
ax.plot(pt200, lin(pt200, *popt))
print(f'Směrnice je:{popt_pcov_to_ufloat(popt, pcov)[0]:.1uP}')
alpha_list.append(popt_pcov_to_ufloat(popt, pcov)[0].n)

ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'peak_pokles.pdf', dpi=300, bbox_inches='tight')


In [ ]:
# exp pokles

def exp_pokles(t, beta, amplit):
    return amplit*np.exp((-1)*beta*t)

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'Maximum', fontsize=20)
ax.set_xlabel(r'$t$ [s]', fontsize=20)

ax.scatter(pt1k, p1k, label=r'$R = 1000\,\Omega$')
popt, pcov= curve_fit(exp_pokles, pt1k, p1k)
pt1k_exp_linspace = np.linspace(np.min(pt1k), np.max(pt1k), 1000)
ax.plot(pt1k_exp_linspace, exp_pokles(pt1k_exp_linspace, *popt), label='Exponenciální fit')
print(f'Tlumící koeficient je:{popt_pcov_to_ufloat(popt, pcov)[1]:.1uP}')

ax.scatter(pt1m, p1m, label=r'$R = 1000000\,\Omega$')
popt, pcov= curve_fit(exp_pokles, pt1m, p1m)
pt1m_exp_linspace = np.linspace(np.min(pt1m), np.max(pt1m), 1000)
ax.plot(pt1m_exp_linspace, exp_pokles(pt1m_exp_linspace, *popt), label='Exponenciální fit')
print(f'Tlumící koeficient je:{popt_pcov_to_ufloat(popt, pcov)[1]:.1uP}')

ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'peak_exp_pokles.pdf', dpi=300, bbox_inches='tight')

In [ ]:
alpha_array = np.array([alpha_list])
alpha_inverse = 1/alpha_array
alpha_inverse = np.ravel(alpha_inverse)
R_odpory_zavislost = np.array([20, 40, 60, 80, 100, 120, 150, 200])

def lin(x, A, B):
    return A*x + B


R_odpory_zavislost_linspace = np.linspace(-70, np.max(R_odpory_zavislost), 1000)

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'$1/\alpha$ [s]', fontsize=20)
ax.set_xlabel(r'$R$ [$\Omega$]', fontsize=20)
ax.scatter(R_odpory_zavislost, alpha_inverse, label='Stanovené hodnoty')
ax.scatter(-R_civka,0, s=100, c='r', label=r'Hodnota $R_C = -40\,\Omega$')

popt, pcov = curve_fit(lin, R_odpory_zavislost, alpha_inverse)
ax.plot(R_odpory_zavislost_linspace, lin(R_odpory_zavislost_linspace, *popt), label='Lineární fit')

ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'alpha_inverse.pdf', dpi=300, bbox_inches='tight')
